# Simple Logistic Regression with PyTorch

## Import

In [2]:
import torch
from torch import nn, optim
from torch.utils.data import TensorDataset,DataLoader
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import numpy as np
import pandas as pd

## Read DataFrame

In [47]:
df = pd.read_csv('./diabetes.csv')
df.head()

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1


## X and y Split

In [48]:
X = df.drop('Outcome',axis=1).values
X[:5]

array([[6.000e+00, 1.480e+02, 7.200e+01, 3.500e+01, 0.000e+00, 3.360e+01,
        6.270e-01, 5.000e+01],
       [1.000e+00, 8.500e+01, 6.600e+01, 2.900e+01, 0.000e+00, 2.660e+01,
        3.510e-01, 3.100e+01],
       [8.000e+00, 1.830e+02, 6.400e+01, 0.000e+00, 0.000e+00, 2.330e+01,
        6.720e-01, 3.200e+01],
       [1.000e+00, 8.900e+01, 6.600e+01, 2.300e+01, 9.400e+01, 2.810e+01,
        1.670e-01, 2.100e+01],
       [0.000e+00, 1.370e+02, 4.000e+01, 3.500e+01, 1.680e+02, 4.310e+01,
        2.288e+00, 3.300e+01]])

In [49]:
num_samples, num_features = X.shape
num_samples, num_features

(768, 8)

In [50]:
y = df['Outcome'].values.reshape(-1,1)

## Train, Validation and Test Split

In [51]:
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.2,random_state=12)

In [52]:
X_train, X_valid, y_train, y_valid = train_test_split(X_train,y_train,test_size=0.1,random_state=12)

## Preprocessing

In [53]:
X_scaler = StandardScaler()
X_train = X_scaler.fit_transform(X_train)
X_valid = X_scaler.transform(X_valid)
X_test = X_scaler.transform(X_test)

## Convert into Tensors

In [54]:
X_train = torch.tensor(X_train,dtype=torch.float32)
X_valid = torch.tensor(X_valid,dtype=torch.float32)
X_test = torch.tensor(X_test,dtype=torch.float32)

In [55]:
y_train = torch.tensor(y_train,dtype=torch.float32)
y_valid = torch.tensor(y_valid,dtype=torch.float32)
y_test = torch.tensor(y_test,dtype=torch.float32)

## Make TensorDataset

In [56]:
train_set = TensorDataset(X_train,y_train)
valid_set = TensorDataset(X_valid,y_valid)
test_set = TensorDataset(X_test,y_test)

## Make DataLoader

In [57]:
train_loader = DataLoader(train_set,batch_size=64,shuffle=True)
valid_loader = DataLoader(valid_set,batch_size=64,shuffle=False)
test_loader = DataLoader(test_set,batch_size=len(train_set),shuffle=False)

## Create Model

In [58]:
model = nn.Sequential(
    nn.Linear(8,1),
    nn.Sigmoid()
)

## Create Optimizer

In [59]:
optimizer = optim.SGD(model.parameters(),lr=0.1,momentum=0.9)

## Train Model and Validate it

In [ ]:
n_epoches = 10
train_hist_loss = []
train_hist_acc = []
valid_hist_loss = []
valid_hist_acc = []

for epoch in range(n_epoches):
    train_mean_loss = 0
    train_correct = 0
    valid_mean_loss = 0
    valid_correct = 0
    num_batches = 0

    for X_batch, y_batch in train_loader:
        y_hat = model(X_batch)
        loss = nn.functional.l1_loss(y_hat, y_batch)
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

        train_mean_loss += loss.item()
        train_correct += torch.sum(y_hat.round() == y_batch).item()
        num_batches += 1

    train_mean_loss = train_mean_loss / num_batches
    train_mean_acc = train_correct / len(train_set)

    train_hist_loss.append(train_mean_loss)
    train_hist_acc.append(train_mean_acc)

    print(f'Epoch {epoch+1}/{n_epoches} | Train Loss: {train_mean_loss:.4f} | Train Accuracy: {train_mean_acc:.4f}')

    with torch.no_grad():
        for X_batch,y_batch in valid_loader:
            y_hat = model(X_batch)
            loss = nn.functional.l1_loss(y_hat,y_batch)
            valid_mean_loss += loss.item()
            valid_correct += torch.sum(y_hat.round() == y_batch).item()
            num_batches += 1

    valid_mean_loss = valid_mean_loss / num_batches
    valid_mean_acc = valid_correct / len(valid_set)

    valid_hist_loss.append(valid_mean_loss)
    valid_hist_acc.append(valid_mean_acc)

    print(f'Epoch {epoch+1}/{n_epoches} | Validation Loss: {valid_mean_loss:.4f} | Validation Accuracy: {valid_mean_acc:.4f}')

Epoch 1/10 | Train Loss: 0.4773 | Train Accuracy: 0.5870
Epoch 1/10 | Validation Loss: 0.0413 | Validation Accuracy: 0.6935
Epoch 2/10 | Train Loss: 0.3661 | Train Accuracy: 0.7246
Epoch 2/10 | Validation Loss: 0.0334 | Validation Accuracy: 0.7258
Epoch 3/10 | Train Loss: 0.3025 | Train Accuracy: 0.7409
Epoch 3/10 | Validation Loss: 0.0306 | Validation Accuracy: 0.7097
Epoch 4/10 | Train Loss: 0.2790 | Train Accuracy: 0.7536
Epoch 4/10 | Validation Loss: 0.0291 | Validation Accuracy: 0.7419
Epoch 5/10 | Train Loss: 0.2675 | Train Accuracy: 0.7681
Epoch 5/10 | Validation Loss: 0.0284 | Validation Accuracy: 0.7581
Epoch 6/10 | Train Loss: 0.2565 | Train Accuracy: 0.7699
Epoch 6/10 | Validation Loss: 0.0279 | Validation Accuracy: 0.7581
Epoch 7/10 | Train Loss: 0.2548 | Train Accuracy: 0.7808
Epoch 7/10 | Validation Loss: 0.0276 | Validation Accuracy: 0.7581
Epoch 8/10 | Train Loss: 0.2521 | Train Accuracy: 0.7772
Epoch 8/10 | Validation Loss: 0.0273 | Validation Accuracy: 0.7581
Epoch 9/

## Create Accuracy function

In [62]:
def accuracy(y_hat, y):
    return torch.sum(y_hat.round() == y).item() / len(y)

## Test the Model

In [63]:
with torch.no_grad():
    X_batch, y_batch = next(iter(test_loader))
    y_hat = model(X_batch)
    loss = nn.functional.l1_loss(y_hat, y_batch)
    acc = accuracy(y_hat, y_batch)

    print(f'Test Loss: {loss:.4f} | Test Accuracy: {acc:.4f}')

Test Loss: 0.2407 | Test Accuracy: 0.7727
